# Day 5 - Hallucination Verification + NLI Metrics
## Trustworthy Evidence-Grounded Framework - FEVEROUS Validation

**Objectives:**
1. Load `facebook/bart-large-mnli` as the NLI / hallucination verifier
2. For each generated justification, split into sentences
3. For each sentence, compute entailment score against every retrieved evidence item
4. Flag sentences whose max entailment score < threshold tau_g = 0.5 as 'unsupported'
5. Compute per-claim and aggregate:
   - **Hallucination rate** = fraction of claims with >= 1 unsupported sentence
   - **Unsupported sentence rate** = fraction of ALL justification sentences that are unsupported
   - **Evidence Grounding Score** = mean max-entailment score across all justification sentences
6. Run across B1, B2, and Proposed systems
7. Save `verification_results.csv`

---
> **Package versions are pinned to match the working Day 1 environment.**

In [1]:
# Cell 1: Install — versions pinned to match Day 1 working environment
!pip install -q \
    "datasets==2.15.0" \
    "huggingface_hub==0.19.4" \
    "fsspec==2023.10.0" \
    transformers \
    accelerate \
    nltk
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
print('Packages ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# Cell 2: Drive mount and paths
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/feverous-validation'
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUT_DIR  = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)
print('Drive mounted')

Mounted at /content/drive
Drive mounted


In [2]:
!pip uninstall -y numpy pandas
!pip install --no-cache-dir numpy==1.26.4 pandas==2.2.2

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 295.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 319.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.19.4 which is incompatible.
sentence-transformers 5.6.0 requires huggingface-hub>=0.23.0, but you have huggingface-hub 0.19.4 which is incompatible.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
tifffile 202

In [2]:
# Cell 3: Imports
import json
import csv
import re
import time
import numpy as np
import torch
import pandas as pd
from nltk.tokenize import sent_tokenize
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm.notebook import tqdm

print('CUDA:', torch.cuda.is_available())

CUDA: True


In [6]:
# Cell 4: Load all data and system outputs
with open(os.path.join(DATA_DIR, 'claims.json'), encoding='utf-8') as f:
    claims = json.load(f)

with open(os.path.join(DATA_DIR, 'evidence_corpus.json'), encoding='utf-8') as f:
    evidence_corpus = json.load(f)

with open(os.path.join(DATA_DIR, 'ers_all_claims_v2.json'), encoding='utf-8') as f:
    ers_all = json.load(f)
claim_id_to_ers = {item['claim_id']: item['ers_ranked'] for item in ers_all}

b1_df = pd.read_csv(os.path.join(OUT_DIR, 'b1_outputs.csv'))
b2_df = pd.read_csv(os.path.join(OUT_DIR, 'b2_outputs.csv'))
pr_df = pd.read_csv(os.path.join(OUT_DIR, 'proposed_outputs_v2.csv'))

retrieval_df = pd.read_csv(os.path.join(OUT_DIR, 'retrieval_results.csv'))
claim_id_to_retrieved = {
    row['claim_id']: json.loads(row['retrieved_idxs'])[:10]
    for _, row in retrieval_df.iterrows()
}

print(f'Claims: {len(claims)}, Corpus: {len(evidence_corpus)}')
print(f'B1 rows: {len(b1_df)}, B2 rows: {len(b2_df)}, Proposed rows: {len(pr_df)}')

Claims: 450, Corpus: 4571
B1 rows: 450, B2 rows: 450, Proposed rows: 450


In [1]:
!pip install -U "huggingface_hub>=0.26.0,<1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.19.4
    Uninstalling huggingface-hub-0.19.4:
      Successfully uninstalled huggingface-hub-0.19.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.


In [3]:
# Cell 5: Load BART-large-MNLI NLI model
NLI_MODEL = 'facebook/bart-large-mnli'
print(f'Loading {NLI_MODEL} ...')

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL)
nli_model = nli_model.to('cuda' if torch.cuda.is_available() else 'cpu')
nli_model.eval()

print(f'NLI model loaded on {"GPU" if torch.cuda.is_available() else "CPU"}')
print('Label mapping:', nli_model.config.id2label)
# IMPORTANT: verify entailment is at index 2
# Expected: {0: 'contradiction', 1: 'neutral', 2: 'entailment'}
ENTAILMENT_IDX = 2
assert nli_model.config.id2label[ENTAILMENT_IDX].lower() == 'entailment', \
    f'Unexpected label order: {nli_model.config.id2label}. Update ENTAILMENT_IDX!'

Loading facebook/bart-large-mnli ...


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

NLI model loaded on GPU
Label mapping: {0: 'contradiction', 1: 'neutral', 2: 'entailment'}


In [7]:
# Cell 6: NLI helpers

ENTAILMENT_THRESHOLD = 0.5    # tau_g: below this = unsupported
MAX_EVIDENCE_PER_CLAIM = 10   # Cap evidence items for NLI to control runtime

import re

def get_entailment_score(premise, hypothesis):
    """
    Run bart-large-mnli on (premise=evidence, hypothesis=claim_sentence).
    Returns the softmax probability for 'entailment'.
    """
    inputs = nli_tokenizer(
        premise, hypothesis,
        return_tensors='pt',
        truncation=True,
        max_length=512
    ).to(nli_model.device)

    with torch.no_grad():
        logits = nli_model(**inputs).logits

    probs = logits.softmax(dim=-1).squeeze().cpu().numpy()
    return float(probs[ENTAILMENT_IDX])


def extract_justification(response_text):
    """Extract justification text after 'Justification:' marker."""
    match = re.search(r'Justification[:\s]+(.*)', str(response_text),
                      re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    return str(response_text).strip()



def strip_citation_markers(text):
    """Remove inline citation markers like '(Evidence 1)' or '(Evidence 1, 3)'
    that confuse the NLI model — they don't appear in the evidence text itself."""
    text = re.sub(r'\(Evidence\s*\d+(?:\s*,\s*\d+)*\)', '', text)
    text = re.sub(r'\[Evidence\s*\d+(?:\s*,\s*\d+)*\]', '', text)
    text = re.sub(r'\s{2,}', ' ', text).strip()
    return text


def verify_justification(justification_text, evidence_texts, threshold=ENTAILMENT_THRESHOLD):
    """
    For each sentence in the justification:
      1. Strip citation markers before scoring
      2. Score against each INDIVIDUAL evidence item (as before)
      3. ALSO score against the CONCATENATION of all evidence items
         (handles sentences that synthesize facts from multiple sources)
      4. Take the max across both approaches as the sentence's grounding score
    """
    justification_text = strip_citation_markers(justification_text)
    sentences = sent_tokenize(justification_text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 15]

    if not sentences or not evidence_texts:
        return []

    # Concatenated premise (capped to avoid runaway length)
    concat_premise = ' '.join(evidence_texts[:MAX_EVIDENCE_PER_CLAIM])

    results = []
    for sent in sentences:
        max_score  = 0.0
        max_ev_idx = -1
        source     = 'individual'

        # Check against individual evidence items
        for j, ev_text in enumerate(evidence_texts[:MAX_EVIDENCE_PER_CLAIM]):
            score = get_entailment_score(premise=ev_text, hypothesis=sent)
            if score > max_score:
                max_score  = score
                max_ev_idx = j

        # Check against the concatenated evidence block
        concat_score = get_entailment_score(premise=concat_premise, hypothesis=sent)
        if concat_score > max_score:
            max_score  = concat_score
            max_ev_idx = -2   # -2 flags "matched via concatenated evidence"
            source     = 'concatenated'

        results.append({
            'sentence':       sent,
            'max_entailment': round(max_score, 4),
            'max_ev_idx':     max_ev_idx,
            'match_source':   source,
            'is_unsupported': int(max_score < threshold)
        })
    return results


print('NLI helpers defined')

NLI helpers defined


In [8]:
# Cell 7: Evidence text getters for each system
# For B1 (no evidence used in generation), we still run NLI against top-10 retrieved
# items for a fair hallucination comparison.

def get_evidence_texts_b1(claim_id):
    idxs = claim_id_to_retrieved.get(claim_id, [])[:MAX_EVIDENCE_PER_CLAIM]
    return [evidence_corpus[i]['text'] for i in idxs if i < len(evidence_corpus)]


def get_evidence_texts_b2(claim_id):
    idxs = claim_id_to_retrieved.get(claim_id, [])[:MAX_EVIDENCE_PER_CLAIM]
    return [evidence_corpus[i]['text'] for i in idxs if i < len(evidence_corpus)]


def get_evidence_texts_proposed(claim_id):
    ers_ranked = claim_id_to_ers.get(claim_id, [])
    return [
        evidence_corpus[item['corpus_idx']]['text']
        for item in ers_ranked[:MAX_EVIDENCE_PER_CLAIM]
        if item['corpus_idx'] < len(evidence_corpus)
    ]


print('Evidence text helpers defined')

Evidence text helpers defined


In [9]:
# Cell 8: Run NLI verification across all 3 systems
# ~2-4 hours on T4. No checkpointing here — keep tab active.

systems_config = [
    ('b1',       b1_df,  get_evidence_texts_b1),
    ('b2',       b2_df,  get_evidence_texts_b2),
    ('proposed', pr_df,  get_evidence_texts_proposed),
]

all_verification_results = []

for system_name, df, get_ev_fn in systems_config:
    print(f'\n=== Verifying {system_name} ({len(df)} claims) ===')

    for _, row in tqdm(df.iterrows(), total=len(df), desc=system_name):
        claim_id   = row['claim_id']
        gold_label = row['gold_label']
        pred_label = row['pred_label']
        correct    = row['correct']
        response   = str(row['response'])

        justification  = extract_justification(response)
        evidence_texts = get_ev_fn(claim_id)
        sentence_results = verify_justification(justification, evidence_texts)

        if sentence_results:
            mean_entailment      = np.mean([s['max_entailment'] for s in sentence_results])
            n_unsupported        = sum(s['is_unsupported'] for s in sentence_results)
            n_sentences          = len(sentence_results)
            claim_has_hallucination = int(n_unsupported > 0)
        else:
            mean_entailment      = 0.0
            n_unsupported        = 0
            n_sentences          = 0
            claim_has_hallucination = 0

        all_verification_results.append({
            'system':            system_name,
            'claim_id':          claim_id,
            'gold_label':        gold_label,
            'pred_label':        pred_label,
            'correct':           correct,
            'justification':     justification,
            'n_sentences':       n_sentences,
            'n_unsupported':     n_unsupported,
            'unsupported_rate':  round(n_unsupported / max(n_sentences, 1), 4),
            'mean_entailment':   round(float(mean_entailment), 4),
            'has_hallucination': claim_has_hallucination,
            'sentence_details':  json.dumps(sentence_results)
        })

print(f'\nVerification complete. Total rows: {len(all_verification_results)}')


=== Verifying b1 (450 claims) ===


b1:   0%|          | 0/450 [00:00<?, ?it/s]


=== Verifying b2 (450 claims) ===


b2:   0%|          | 0/450 [00:00<?, ?it/s]


=== Verifying proposed (450 claims) ===


proposed:   0%|          | 0/450 [00:00<?, ?it/s]


Verification complete. Total rows: 1350


In [10]:
# Cell 9: Save verification_results.csv
verif_path = os.path.join(OUT_DIR, 'verification_results_v2.csv')
with open(verif_path, 'w', newline='', encoding='utf-8') as f:
    if all_verification_results:
        writer = csv.DictWriter(f, fieldnames=list(all_verification_results[0].keys()))
        writer.writeheader()
        writer.writerows(all_verification_results)
print(f'Saved: {verif_path}')
print(f'Rows: {len(all_verification_results)} (expected {3 * len(claims)} = 3 x {len(claims)})')

Saved: /content/drive/MyDrive/feverous-validation/outputs/verification_results_v2.csv
Rows: 1350 (expected 1350 = 3 x 450)


In [11]:
# Cell 10: Print hallucination and grounding metrics
verif_df = pd.read_csv(verif_path)

print('=' * 65)
print('HALLUCINATION AND GROUNDING METRICS')
print('=' * 65)

for system in ['b1', 'b2', 'proposed']:
    sub = verif_df[verif_df['system'] == system]
    halluc_rate     = sub['has_hallucination'].mean()
    unsup_sent_rate = sub['unsupported_rate'].mean()
    grounding_score = sub['mean_entailment'].mean()
    label_acc       = sub['correct'].mean()
    sys_label = {'b1': 'B1 (LLM-only)', 'b2': 'B2 (Standard RAG)', 'proposed': 'Proposed (ERS)'}[system]

    print(f'\n{sys_label}:')
    print(f'  Label Accuracy:           {label_acc:.4f}')
    print(f'  Hallucination Rate:       {halluc_rate:.4f}  (>= 1 unsupported sentence)')
    print(f'  Unsupported Sent. Rate:   {unsup_sent_rate:.4f}  (fraction of sentences)')
    print(f'  Evidence Grounding Score: {grounding_score:.4f}  (mean max-entailment)')

print('\n' + '=' * 65)

HALLUCINATION AND GROUNDING METRICS

B1 (LLM-only):
  Label Accuracy:           0.3889
  Hallucination Rate:       0.8222  (>= 1 unsupported sentence)
  Unsupported Sent. Rate:   0.7439  (fraction of sentences)
  Evidence Grounding Score: 0.2807  (mean max-entailment)

B2 (Standard RAG):
  Label Accuracy:           0.4378
  Hallucination Rate:       0.8378  (>= 1 unsupported sentence)
  Unsupported Sent. Rate:   0.7404  (fraction of sentences)
  Evidence Grounding Score: 0.2786  (mean max-entailment)

Proposed (ERS):
  Label Accuracy:           0.4600
  Hallucination Rate:       0.8533  (>= 1 unsupported sentence)
  Unsupported Sent. Rate:   0.7439  (fraction of sentences)
  Evidence Grounding Score: 0.2766  (mean max-entailment)



In [12]:
# Cell 11: Derive confidence scores for ECE calibration in Day 6
# Confidence = mean_entailment (already in [0,1])

for system_name, out_filename in [('proposed', 'proposed_with_confidence.csv'),
                                   ('b2',       'b2_with_confidence.csv')]:
    sub = verif_df[verif_df['system'] == system_name].copy()
    sub['confidence'] = sub['mean_entailment']
    conf_path = os.path.join(OUT_DIR, out_filename)
    sub[['claim_id', 'gold_label', 'pred_label', 'correct',
         'mean_entailment', 'confidence', 'has_hallucination']].to_csv(conf_path, index=False)
    print(f'Saved: {conf_path}')

Saved: /content/drive/MyDrive/feverous-validation/outputs/proposed_with_confidence.csv
Saved: /content/drive/MyDrive/feverous-validation/outputs/b2_with_confidence.csv


## Day 5 Complete

Files saved:
- `outputs/verification_results.csv` — NLI entailment scores, hallucination flags, grounding scores (1,350 rows)
- `outputs/proposed_with_confidence.csv` — Proposed system with confidence scores (for Day 6 ECE)
- `outputs/b2_with_confidence.csv` — B2 with confidence scores (for Day 6 comparison)

Next: Open `Day6_Calibration_Ablation.ipynb`

In [9]:
# ================================================================
# EXPERIMENT — ERS + CONSTRAINED GENERATION + NLI REGENERATION
# 50 CLAIM PILOT
# ================================================================
#
# This is a NEW EXPERIMENT.
#
# It does NOT modify:
#   b1_df
#   b2_df
#   pr_df
#   all_verification_results
#
# It creates:
#   proposed_nli_50_results
#
# ================================================================

import os
import json
import re
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# ================================================================
# CONFIGURATION
# ================================================================

TEST_N = 50

GENERATION_K = 3

MAX_ATTEMPTS = 2

GENERATION_MAX_NEW_TOKENS = 100

GROUNDING_THRESHOLD = 0.5

# ================================================================
# IMPORTANT:
# This assumes your existing generation model/tokenizer are:
#
#   model
#   tokenizer
#
# If your notebook uses different variable names, change ONLY
# these two lines.
# ================================================================

GEN_MODEL = nli_model
GEN_TOKENIZER = nli_tokenizer


# ================================================================
# GET TOP-K ERS EVIDENCE
# ================================================================

def get_generation_evidence_proposed(claim_id, k=GENERATION_K):

    ers_ranked = claim_id_to_ers.get(claim_id, [])

    evidence = []

    for item in ers_ranked[:k]:

        corpus_idx = item['corpus_idx']

        if corpus_idx < len(evidence_corpus):

            evidence.append({
                "corpus_idx": corpus_idx,
                "text": evidence_corpus[corpus_idx]['text'],
                "ers": item.get('ers', None),
                "relevance": item.get('relevance', None),
                "consistency": item.get('consistency', None)
            })

    return evidence


# ================================================================
# BUILD STRICT PROMPT
# ================================================================

def build_grounded_prompt(
    claim,
    evidence_items,
    previous_response=None
):

    evidence_block = ""

    for i, item in enumerate(evidence_items):

        evidence_block += (
            f"\n[EVIDENCE {i+1}]\n"
            f"{item['text']}\n"
        )

    retry_instruction = ""

    if previous_response is not None:

        retry_instruction = """
IMPORTANT — YOUR PREVIOUS ANSWER WAS NOT FULLY GROUNDED.

Rewrite the answer.

Remove every factual statement that is not directly supported
by the evidence.

Do NOT add new information.

Do NOT use outside knowledge.

Do NOT make assumptions.

Keep the justification short.
"""


    prompt = f"""
You are an evidence-grounded claim verification system.

Your ONLY source of factual information is the evidence provided below.

You MUST NOT use outside knowledge.

You MUST NOT introduce:
- facts
- dates
- numbers
- names
- locations
- relationships
- explanations

unless they are explicitly supported by the provided evidence.

The claim must be classified as:

SUPPORTS
REFUTES
or
NEI

Rules:

1. Use ONLY the provided evidence.
2. Every factual statement in the justification must be supported
   by one or more evidence items.
3. Do not infer additional facts.
4. Do not use your pretrained/world knowledge.
5. Do not add background information.
6. If the evidence is insufficient, use NEI.
7. Keep the justification to 1–2 short sentences.
8. Every factual sentence MUST include an evidence citation.
9. Do not mention evidence that does not support the sentence.

Return EXACTLY:

VERDICT: <SUPPORTS|REFUTES|NEI>
JUSTIFICATION: <short evidence-grounded explanation>

Example:

VERDICT: REFUTES
JUSTIFICATION: The evidence states that four structures are
associated with the adductor hiatus, contradicting the claim's
statement that nine structures are associated with it. [EVIDENCE 1]

{retry_instruction}

====================
EVIDENCE
====================
{evidence_block}

====================
CLAIM
====================
{claim}
"""

    return prompt


# ================================================================
# GENERATION FUNCTION
# ================================================================

def generate_with_mistral(prompt):

    inputs = GEN_TOKENIZER(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(GEN_MODEL.device)

    with torch.no_grad():

        outputs = GEN_MODEL.generate(
            **inputs,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            pad_token_id=GEN_TOKENIZER.eos_token_id
        )

    # Only decode newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = GEN_TOKENIZER.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


# ================================================================
# EXTRACT JUSTIFICATION
# ================================================================

def extract_verdict(response):

    match = re.search(
        r'VERDICT\s*:\s*(SUPPORTS|REFUTES|NEI)',
        response,
        re.IGNORECASE
    )

    if match:
        return match.group(1).upper()

    return None


# ================================================================
# EXTRACT JUSTIFICATION
# ================================================================

def extract_strict_justification(response):

    match = re.search(
        r'JUSTIFICATION\s*:\s*(.*)',
        response,
        re.IGNORECASE | re.DOTALL
    )

    if match:

        text = match.group(1).strip()

        return text

    return response.strip()


# ================================================================
# CHECK EVIDENCE CITATIONS
# ================================================================

def count_evidence_citations(text):

    citations = re.findall(
        r'\[EVIDENCE\s+\d+(?:\s*,\s*\d+)*\]',
        text,
        re.IGNORECASE
    )

    return len(citations)


# ================================================================
# RUN ONE CLAIM
# ================================================================

def run_grounded_claim(claim_obj):

    claim_id = claim_obj["claim_id"]

    claim_text = claim_obj["claim"]

    gold_label = claim_obj.get("label", None)

    evidence_items = get_generation_evidence_proposed(
        claim_id,
        k=GENERATION_K
    )

    evidence_texts = [
        item["text"]
        for item in evidence_items
    ]

    if not evidence_texts:

        return {
            "claim_id": claim_id,
            "claim": claim_text,
            "gold_label": gold_label,
            "attempts": 0,
            "success": False,
            "reason": "no_evidence",
            "response": "",
            "verdict": None,
            "justification": "",
            "n_sentences": 0,
            "n_unsupported": 0,
            "unsupported_rate": 0.0,
            "mean_entailment": 0.0,
            "has_hallucination": 0,
            "citation_count": 0
        }


    previous_response = None

    final_response = ""

    final_justification = ""

    final_sentence_results = []

    accepted = False

    attempts_used = 0


    # ============================================================
    # REGENERATION LOOP
    # ============================================================

    for attempt in range(MAX_ATTEMPTS):

        attempts_used = attempt + 1

        prompt = build_grounded_prompt(
            claim=claim_text,
            evidence_items=evidence_items,
            previous_response=previous_response
        )

        response = generate_with_mistral(prompt)

        justification = extract_strict_justification(
            response
        )

        # --------------------------------------------------------
        # Run existing NLI verifier
        # --------------------------------------------------------

        sentence_results = verify_justification(
            justification,
            evidence_texts,
            threshold=GROUNDING_THRESHOLD
        )

        final_response = response

        final_justification = justification

        final_sentence_results = sentence_results

        # --------------------------------------------------------
        # Check grounding
        # --------------------------------------------------------

        if sentence_results:

            unsupported = sum(
                s["is_unsupported"]
                for s in sentence_results
            )

            unsupported_rate = (
                unsupported /
                len(sentence_results)
            )

        else:

            unsupported = 0
            unsupported_rate = 1.0


        # --------------------------------------------------------
        # ACCEPT ONLY FULLY GROUNDED JUSTIFICATION
        # --------------------------------------------------------

        if (
            sentence_results
            and
            unsupported == 0
        ):

            accepted = True

            break


        # --------------------------------------------------------
        # Prepare for regeneration
        # --------------------------------------------------------

        previous_response = response


    # ============================================================
    # FINAL METRICS
    # ============================================================

    if final_sentence_results:

        n_sentences = len(final_sentence_results)

        n_unsupported = sum(
            s["is_unsupported"]
            for s in final_sentence_results
        )

        unsupported_rate = (
            n_unsupported /
            n_sentences
        )

        mean_entailment = np.mean([
            s["max_entailment"]
            for s in final_sentence_results
        ])

        has_hallucination = int(
            n_unsupported > 0
        )

    else:

        n_sentences = 0
        n_unsupported = 0
        unsupported_rate = 0.0
        mean_entailment = 0.0
        has_hallucination = 1


    verdict = extract_verdict(final_response)

    citation_count = count_evidence_citations(
        final_justification
    )


    return {

        "claim_id": claim_id,

        "claim": claim_text,

        "gold_label": gold_label,

        "attempts": attempts_used,

        "accepted_after_nli": accepted,

        "response": final_response,

        "verdict": verdict,

        "justification": final_justification,

        "evidence_count": len(evidence_texts),

        "evidence_indices": json.dumps([
            x["corpus_idx"]
            for x in evidence_items
        ]),

        "ers_scores": json.dumps([
            x["ers"]
            for x in evidence_items
        ]),

        "citation_count": citation_count,

        "n_sentences": n_sentences,

        "n_unsupported": n_unsupported,

        "unsupported_rate": round(
            unsupported_rate,
            4
        ),

        "mean_entailment": round(
            float(mean_entailment),
            4
        ),

        "has_hallucination": has_hallucination,

        "sentence_details": json.dumps(
            final_sentence_results
        )
    }


# ================================================================
# RUN 50 CLAIMS
# ================================================================

print("=" * 80)
print("ERS + CONSTRAINED GENERATION + NLI REGENERATION")
print(f"Running {TEST_N} claims")
print(f"ERS evidence K = {GENERATION_K}")
print(f"Maximum attempts = {MAX_ATTEMPTS}")
print(f"NLI threshold = {GROUNDING_THRESHOLD}")
print("=" * 80)


pilot_claims = claims[:TEST_N]

proposed_nli_50_results = []


for i, claim_obj in enumerate(
    tqdm(
        pilot_claims,
        desc="50-claim grounded generation"
    )
):

    try:

        result = run_grounded_claim(
            claim_obj
        )

        result["claim_index"] = i

        proposed_nli_50_results.append(
            result
        )

    except Exception as e:

        print(
            f"\nERROR on claim {i}: {e}"
        )

        proposed_nli_50_results.append({

            "claim_index": i,

            "claim_id": claim_obj.get(
                "claim_id"
            ),

            "claim": claim_obj.get(
                "claim"
            ),

            "gold_label": claim_obj.get(
                "label"
            ),

            "error": str(e),

            "has_hallucination": 1
        })


# ================================================================
# DATAFRAME
# ================================================================

pilot_df = pd.DataFrame(
    proposed_nli_50_results
)


# ================================================================
# SAVE
# ================================================================

pilot_path = os.path.join(
    OUT_DIR,
    "proposed_nli_regeneration_50.csv"
)

pilot_df.to_csv(
    pilot_path,
    index=False,
    encoding="utf-8"
)

print(
    f"\nSaved: {pilot_path}"
)


# ================================================================
# SUMMARY
# ================================================================

valid = pilot_df[
    pilot_df.get(
        "response",
        pd.Series(dtype=str)
    ).notna()
].copy()


print("\n" + "=" * 80)
print("50-CLAIM PILOT RESULTS")
print("=" * 80)

print(
    "Claims processed:",
    len(pilot_df)
)

print(
    "Claims accepted after NLI:",
    pilot_df.get(
        "accepted_after_nli",
        pd.Series(dtype=bool)
    ).sum()
)


if len(valid) > 0:

    hallucination_rate = (
        valid["has_hallucination"]
        .mean()
    )

    unsupported_sentence_rate = (
        valid["unsupported_rate"]
        .mean()
    )

    grounding_score = (
        valid["mean_entailment"]
        .mean()
    )

    avg_attempts = (
        valid["attempts"]
        .mean()
    )

    avg_sentences = (
        valid["n_sentences"]
        .mean()
    )

    print(
        f"\nHallucination Rate: "
        f"{hallucination_rate:.4f}"
    )

    print(
        f"Unsupported Sentence Rate: "
        f"{unsupported_sentence_rate:.4f}"
    )

    print(
        f"Grounding Score: "
        f"{grounding_score:.4f}"
    )

    print(
        f"Average Attempts: "
        f"{avg_attempts:.2f}"
    )

    print(
        f"Average Sentences: "
        f"{avg_sentences:.2f}"
    )


# ================================================================
# COMPARE WITH EXISTING 50 CLAIMS
# ================================================================

print("\n" + "=" * 80)
print("COMPARISON WITH EXISTING SYSTEM")
print("=" * 80)


# Existing proposed results for the same first 50 claims
existing_50 = verif_df[
    (verif_df["system"] == "proposed") &
    (verif_df["claim_id"].isin(
        [c["claim_id"] for c in pilot_claims]
    ))
].copy()


if len(existing_50) > 0:

    print("\nExisting Proposed ERS:")

    print(
        f"  Hallucination Rate: "
        f"{existing_50['has_hallucination'].mean():.4f}"
    )

    print(
        f"  Unsupported Sentence Rate: "
        f"{existing_50['unsupported_rate'].mean():.4f}"
    )

    print(
        f"  Grounding Score: "
        f"{existing_50['mean_entailment'].mean():.4f}"
    )


if len(valid) > 0:

    print("\nNew ERS + NLI Regeneration:")

    print(
        f"  Hallucination Rate: "
        f"{valid['has_hallucination'].mean():.4f}"
    )

    print(
        f"  Unsupported Sentence Rate: "
        f"{valid['unsupported_rate'].mean():.4f}"
    )

    print(
        f"  Grounding Score: "
        f"{valid['mean_entailment'].mean():.4f}"
    )


# ================================================================
# PER-CLAIM SUMMARY
# ================================================================

print("\n" + "=" * 80)
print("PER-CLAIM RESULTS")
print("=" * 80)

for _, r in pilot_df.iterrows():

    print(
        f"Claim {int(r['claim_index']):02d} | "
        f"attempts={r.get('attempts', '-') } | "
        f"accepted={r.get('accepted_after_nli', '-') } | "
        f"sentences={r.get('n_sentences', '-') } | "
        f"unsupported={r.get('n_unsupported', '-') } | "
        f"grounding={r.get('mean_entailment', 0):.3f} | "
        f"hallucinated={r.get('has_hallucination', '-')}"
    )

print("\n" + "=" * 80)
print("EXPERIMENT COMPLETE")
print("=" * 80)

ERS + CONSTRAINED GENERATION + NLI REGENERATION
Running 50 claims
ERS evidence K = 3
Maximum attempts = 2
NLI threshold = 0.5


50-claim grounded generation:   0%|          | 0/50 [00:00<?, ?it/s]


ERROR on claim 0: 'BartForSequenceClassification' object has no attribute 'generate'

ERROR on claim 1: 'BartForSequenceClassification' object has no attribute 'generate'

ERROR on claim 2: 'BartForSequenceClassification' object has no attribute 'generate'

ERROR on claim 3: 'BartForSequenceClassification' object has no attribute 'generate'

ERROR on claim 4: 'BartForSequenceClassification' object has no attribute 'generate'

ERROR on claim 5: 'BartForSequenceClassification' object has no attribute 'generate'

ERROR on claim 6: 'BartForSequenceClassification' object has no attribute 'generate'

ERROR on claim 7: 'BartForSequenceClassification' object has no attribute 'generate'

ERROR on claim 8: 'BartForSequenceClassification' object has no attribute 'generate'

ERROR on claim 9: 'BartForSequenceClassification' object has no attribute 'generate'

ERROR on claim 10: 'BartForSequenceClassification' object has no attribute 'generate'

ERROR on claim 11: 'BartForSequenceClassification' o

/tmp/ipykernel_716/1060285769.py:609: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  valid = pilot_df[


IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).

In [10]:
print("nli_model:", type(nli_model))

# Find likely generation model variables
for name in [
    "model",
    "llm_model",
    "generation_model",
    "mistral_model",
    "generator"
]:
    if name in globals():
        print(name, "->", type(globals()[name]))

nli_model: <class 'transformers.models.bart.modeling_bart.BartForSequenceClassification'>


In [13]:
# ================================================================
# REVISED HALLUCINATION / GROUNDING VERIFICATION
# ================================================================
#
# Key changes:
# 1. Never concatenate all evidence into one giant NLI premise.
# 2. Long evidence is chunked before NLI evaluation.
# 3. Each generated sentence is compared against every evidence chunk.
# 4. max entailment is used as the support score.
# 5. Claim-level hallucination != sentence-level unsupported rate.
# 6. Unsupported sentence rate uses global sentence counts.
#
# ================================================================

import re
import json
import math
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ----------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------

NLI_MODEL_NAME = "facebook/bart-large-mnli"

# Main threshold used for final reported metric.
ENTAILMENT_THRESHOLD = 0.50

# Threshold sensitivity analysis.
THRESHOLDS = [0.30, 0.40, 0.50, 0.60, 0.70]

# Evidence chunk size.
# Keep this comfortably below BART's 512-token maximum.
EVIDENCE_CHUNK_TOKENS = 180

# Maximum number of evidence chunks evaluated per claim.
# Increase if GPU memory allows.
MAX_EVIDENCE_CHUNKS = 40

# Number of examples to print for manual inspection.
N_DIAGNOSTIC_EXAMPLES = 20

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Loading:", NLI_MODEL_NAME)

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME
).to(DEVICE)

nli_model.eval()

# ----------------------------------------------------------------
# CHECK LABEL MAPPING
# ----------------------------------------------------------------

print("NLI labels:", nli_model.config.id2label)

# BART MNLI normally uses:
# 0 = contradiction
# 1 = neutral
# 2 = entailment

ENTAILMENT_IDX = None
CONTRADICTION_IDX = None
NEUTRAL_IDX = None

for idx, label in nli_model.config.id2label.items():
    label_lower = label.lower()

    if label_lower == "entailment":
        ENTAILMENT_IDX = int(idx)
    elif label_lower == "contradiction":
        CONTRADICTION_IDX = int(idx)
    elif label_lower == "neutral":
        NEUTRAL_IDX = int(idx)

assert ENTAILMENT_IDX is not None, "Could not find entailment label."

print("ENTAILMENT_IDX:", ENTAILMENT_IDX)
print("CONTRADICTION_IDX:", CONTRADICTION_IDX)
print("NEUTRAL_IDX:", NEUTRAL_IDX)


# ================================================================
# SENTENCE SPLITTING
# ================================================================

def split_sentences(text):
    """
    Conservative sentence splitting for generated justifications.
    """

    if text is None:
        return []

    text = str(text).strip()

    if not text:
        return []

    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text)

    # Basic sentence segmentation.
    # Handles ., !, ?, followed by whitespace/capital/number.
    sentences = re.split(
        r"(?<=[.!?])\s+(?=[A-Z0-9])",
        text
    )

    sentences = [
        s.strip()
        for s in sentences
        if s.strip()
    ]

    return sentences


# ================================================================
# EVIDENCE CHUNKING
# ================================================================

def chunk_text_by_tokens(
    text,
    tokenizer,
    max_tokens=EVIDENCE_CHUNK_TOKENS
):
    """
    Split one evidence document into token-limited chunks.

    We intentionally use a much smaller size than 512 so the
    hypothesis + special tokens also fit comfortably.
    """

    if text is None:
        return []

    text = str(text).strip()

    if not text:
        return []

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    if not token_ids:
        return []

    chunks = []

    for start in range(0, len(token_ids), max_tokens):

        chunk_ids = token_ids[
            start:start + max_tokens
        ]

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        ).strip()

        if chunk_text:
            chunks.append(chunk_text)

    return chunks


def build_evidence_chunks(
    evidence_texts,
    max_chunks=MAX_EVIDENCE_CHUNKS
):
    """
    Convert evidence documents into manageable NLI chunks.
    """

    all_chunks = []

    for evidence_idx, evidence_text in enumerate(
        evidence_texts
    ):

        chunks = chunk_text_by_tokens(
            evidence_text,
            nli_tokenizer
        )

        for chunk_idx, chunk in enumerate(chunks):

            all_chunks.append({
                "evidence_idx": evidence_idx,
                "chunk_idx": chunk_idx,
                "text": chunk
            })

            if len(all_chunks) >= max_chunks:
                return all_chunks

    return all_chunks


# ================================================================
# NLI SCORING
# ================================================================

@torch.inference_mode()
def nli_entailment_score(
    premise,
    hypothesis
):
    """
    Compute P(entailment).

    premise    = evidence
    hypothesis = generated justification sentence
    """

    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
    }

    outputs = nli_model(**inputs)

    probs = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    return float(
        probs[ENTAILMENT_IDX].detach().cpu()
    )


# ================================================================
# VERIFY ONE SENTENCE
# ================================================================

def verify_sentence(
    sentence,
    evidence_chunks
):
    """
    Check whether at least one evidence chunk entails the
    generated sentence.

    Returns the strongest supporting evidence and score.
    """

    if not evidence_chunks:
        return {
            "sentence": sentence,
            "max_entailment": 0.0,
            "supported": False,
            "best_evidence_idx": None,
            "best_chunk_idx": None,
            "best_evidence": None
        }

    best_score = -1.0
    best_item = None

    for item in evidence_chunks:

        score = nli_entailment_score(
            premise=item["text"],
            hypothesis=sentence
        )

        if score > best_score:

            best_score = score

            best_item = item

    return {
        "sentence": sentence,
        "max_entailment": best_score,
        "supported": best_score >= ENTAILMENT_THRESHOLD,
        "best_evidence_idx": (
            best_item["evidence_idx"]
            if best_item else None
        ),
        "best_chunk_idx": (
            best_item["chunk_idx"]
            if best_item else None
        ),
        "best_evidence": (
            best_item["text"]
            if best_item else None
        )
    }


# ================================================================
# VERIFY ONE JUSTIFICATION
# ================================================================

def verify_justification(
    justification,
    evidence_texts
):

    sentences = split_sentences(
        justification
    )

    evidence_chunks = build_evidence_chunks(
        evidence_texts
    )

    sentence_results = []

    for sentence in sentences:

        result = verify_sentence(
            sentence,
            evidence_chunks
        )

        sentence_results.append(
            result
        )

    n_sentences = len(sentence_results)

    n_unsupported = sum(
        1
        for r in sentence_results
        if not r["supported"]
    )

    entailment_scores = [
        r["max_entailment"]
        for r in sentence_results
    ]

    if n_sentences > 0:

        mean_entailment = float(
            np.mean(entailment_scores)
        )

        # This is the correct sentence-level metric:
        unsupported_sentence_rate = (
            n_unsupported / n_sentences
        )

    else:

        mean_entailment = 0.0
        unsupported_sentence_rate = 0.0

    # Claim-level hallucination:
    # ONE unsupported sentence => claim considered hallucinated.
    claim_hallucinated = (
        n_unsupported > 0
        if n_sentences > 0
        else False
    )

    return {
        "sentences": sentence_results,
        "n_sentences": n_sentences,
        "n_unsupported": n_unsupported,
        "mean_entailment": mean_entailment,
        "unsupported_sentence_rate": unsupported_sentence_rate,
        "claim_hallucinated": claim_hallucinated
    }


# ================================================================
# DATASET-LEVEL EVALUATION
# ================================================================

def evaluate_hallucination(
    outputs,
    method_name,
    threshold=ENTAILMENT_THRESHOLD,
    diagnostic=False
):
    """
    outputs should contain one row/dict per claim.

    Expected fields should include something equivalent to:
      - justification
      - evidence_texts

    If your existing structure uses different names, change the
    two field accesses below.
    """

    global ENTAILMENT_THRESHOLD

    old_threshold = ENTAILMENT_THRESHOLD
    ENTAILMENT_THRESHOLD = threshold

    claim_results = []

    total_sentences = 0
    total_unsupported = 0

    for i, item in enumerate(outputs):

        # --------------------------------------------------------
        # ADAPT THESE TWO LINES IF YOUR JSON STRUCTURE DIFFERS
        # --------------------------------------------------------

        justification = (
            item.get("justification", "")
            or item.get("explanation", "")
            or item.get("rationale", "")
        )

        evidence_texts = (
            item.get("evidence_texts", [])
            or item.get("evidence", [])
            or []
        )

        # Make evidence strings if evidence is stored as dicts.
        cleaned_evidence = []

        for ev in evidence_texts:

            if isinstance(ev, str):
                cleaned_evidence.append(ev)

            elif isinstance(ev, dict):

                text = (
                    ev.get("text")
                    or ev.get("evidence_text")
                    or ev.get("content")
                    or ""
                )

                if text:
                    cleaned_evidence.append(text)

        verification = verify_justification(
            justification=justification,
            evidence_texts=cleaned_evidence
        )

        result = {
            "claim_index": i,
            "method": method_name,
            "justification": justification,
            "n_sentences": verification["n_sentences"],
            "n_unsupported": verification["n_unsupported"],
            "mean_entailment": verification["mean_entailment"],
            "unsupported_sentence_rate": (
                verification["unsupported_sentence_rate"]
            ),
            "claim_hallucinated": (
                verification["claim_hallucinated"]
            ),
            "sentence_results": (
                verification["sentences"]
            )
        }

        claim_results.append(result)

        total_sentences += verification["n_sentences"]
        total_unsupported += verification["n_unsupported"]

        # --------------------------------------------------------
        # DIAGNOSTIC OUTPUT
        # --------------------------------------------------------

        if diagnostic and i < N_DIAGNOSTIC_EXAMPLES:

            print("\n" + "=" * 80)
            print(f"{method_name} | CLAIM {i}")
            print("=" * 80)

            print("\nJUSTIFICATION:")
            print(justification)

            print("\nSENTENCE VERIFICATION:")

            for sr in verification["sentences"]:

                print("\nSentence:")
                print(sr["sentence"])

                print(
                    f"Entailment: "
                    f"{sr['max_entailment']:.4f}"
                )

                print(
                    "Supported:",
                    sr["supported"]
                )

                print(
                    "Best evidence:"
                )

                print(
                    sr["best_evidence"][:500]
                    if sr["best_evidence"]
                    else "NONE"
                )

    # Restore threshold
    ENTAILMENT_THRESHOLD = old_threshold

    # ------------------------------------------------------------
    # CORRECT AGGREGATION
    # ------------------------------------------------------------

    n_claims = len(claim_results)

    hallucinated_claims = sum(
        r["claim_hallucinated"]
        for r in claim_results
    )

    hallucination_rate = (
        hallucinated_claims / n_claims
        if n_claims > 0
        else 0.0
    )

    # IMPORTANT:
    # This is GLOBAL sentence-level aggregation.
    unsupported_sentence_rate = (
        total_unsupported / total_sentences
        if total_sentences > 0
        else 0.0
    )

    grounding_score = (
        float(np.mean([
            r["mean_entailment"]
            for r in claim_results
        ]))
        if claim_results
        else 0.0
    )

    return {
        "method": method_name,
        "threshold": threshold,
        "n_claims": n_claims,
        "hallucination_rate": hallucination_rate,
        "unsupported_sentence_rate": (
            unsupported_sentence_rate
        ),
        "grounding_score": grounding_score,
        "total_sentences": total_sentences,
        "total_unsupported_sentences": (
            total_unsupported
        ),
        "claim_results": claim_results
    }


# ================================================================
# THRESHOLD SENSITIVITY
# ================================================================

def threshold_sensitivity(
    outputs,
    method_name,
    thresholds=THRESHOLDS
):

    rows = []

    for threshold in thresholds:

        print(
            f"\nEvaluating {method_name} "
            f"at threshold={threshold}"
        )

        result = evaluate_hallucination(
            outputs=outputs,
            method_name=method_name,
            threshold=threshold,
            diagnostic=False
        )

        rows.append({
            "method": method_name,
            "threshold": threshold,
            "hallucination_rate": (
                result["hallucination_rate"]
            ),
            "unsupported_sentence_rate": (
                result["unsupported_sentence_rate"]
            ),
            "grounding_score": (
                result["grounding_score"]
            )
        })

    return pd.DataFrame(rows)


# ================================================================
# EXAMPLE:
# RUN ON YOUR EXISTING OUTPUT OBJECTS
# ================================================================

# Replace these variable names ONLY if your notebook uses
# different names.

# proposed_verification = evaluate_hallucination(
#     proposed_outputs,
#     "proposed",
#     threshold=0.50,
#     diagnostic=True
# )

# b2_verification = evaluate_hallucination(
#     b2_outputs,
#     "b2",
#     threshold=0.50,
#     diagnostic=True
# )

# b1_verification = evaluate_hallucination(
#     b1_outputs,
#     "b1",
#     threshold=0.50,
#     diagnostic=True
# )


# ================================================================
# PRINT RESULTS
# ================================================================

def print_verification_summary(result):

    print("\n" + "=" * 70)
    print(f"HALLUCINATION METRICS — {result['method']}")
    print("=" * 70)

    print(
        f"Claims:                    "
        f"{result['n_claims']}"
    )

    print(
        f"Threshold:                 "
        f"{result['threshold']:.2f}"
    )

    print(
        f"Hallucinated claims:       "
        f"{result['hallucination_rate']:.4f}"
    )

    print(
        f"Unsupported sentence rate: "
        f"{result['unsupported_sentence_rate']:.4f}"
    )

    print(
        f"Grounding score:           "
        f"{result['grounding_score']:.4f}"
    )

    print(
        f"Total sentences:           "
        f"{result['total_sentences']}"
    )

    print(
        f"Unsupported sentences:     "
        f"{result['total_unsupported_sentences']}"
    )

Device: cuda
Loading: facebook/bart-large-mnli


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


NLI labels: {0: 'contradiction', 1: 'neutral', 2: 'entailment'}
ENTAILMENT_IDX: 2
CONTRADICTION_IDX: 0
NEUTRAL_IDX: 1


In [14]:
print("Variables containing output/results:")

for name in sorted(globals()):
    if any(x in name.lower() for x in [
        "output",
        "result",
        "claim",
        "proposed",
        "b1",
        "b2"
    ]):
        print(name)

Variables containing output/results:
MAX_EVIDENCE_PER_CLAIM
all_verification_results
b1_df
b2_df
claim_has_hallucination
claim_id
claim_id_to_ers
claim_id_to_retrieved
claims
get_evidence_texts_b1
get_evidence_texts_b2
get_evidence_texts_proposed
sentence_results


In [18]:
# ================================================================
# EXPERIMENT ONLY — REVISED VERIFICATION ON 10 CLAIMS
# ================================================================
#
# IMPORTANT:
# This does NOT modify your existing:
#   b1_df
#   b2_df
#   claims
#   all_verification_results
#
# It only creates:
#   revised_test_results
#
# ================================================================

TEST_N = 10

revised_test_results = []

print("=" * 80)
print("RUNNING REVISED VERIFICATION ON 10 CLAIMS")
print("=" * 80)

for i in range(min(TEST_N, len(claims))):

    claim = claims[i]

    # ------------------------------------------------------------
    # Get CLAIM ID
    # ------------------------------------------------------------

    if isinstance(claim, dict):

        claim_id_value = (
            claim.get("claim_id")
            or claim.get("id")
            or claim.get("claimId")
        )

    else:
        claim_id_value = claim

    print(
        f"\nProcessing claim {i} | "
        f"claim_id = {claim_id_value}"
    )

    # ------------------------------------------------------------
    # GET PROPOSED EVIDENCE
    #
    # Your existing function expects claim_id.
    # ------------------------------------------------------------

    try:

        evidence_texts = get_evidence_texts_proposed(
            claim_id_value
        )

    except Exception as e:

        print(
            f"ERROR getting evidence: {e}"
        )

        evidence_texts = []

    print(
        f"Evidence documents: {len(evidence_texts)}"
    )

    # ------------------------------------------------------------
    # FIND EXISTING PROPOSED JUSTIFICATION
    # ------------------------------------------------------------

    justification = None

    # First search all_verification_results
    for r in all_verification_results:

        if not isinstance(r, dict):
            continue

        rid = (
            r.get("claim_id")
            or r.get("id")
        )

        if rid == claim_id_value:

            for key in [
                "justification",
                "explanation",
                "rationale",
                "generated_justification"
            ]:

                if r.get(key):
                    justification = r[key]
                    break

            if justification:
                break

    # If not found there, search the claim itself
    if justification is None and isinstance(claim, dict):

        for key in [
            "proposed_justification",
            "justification",
            "explanation",
            "rationale",
            "proposed_explanation"
        ]:

            if claim.get(key):
                justification = claim[key]
                break

    # ------------------------------------------------------------
    # STOP IF WE CANNOT FIND JUSTIFICATION
    # ------------------------------------------------------------

    if not justification:

        print(
            "WARNING: Could not find proposed justification."
        )

        revised_test_results.append({
            "claim_index": i,
            "claim_id": claim_id_value,
            "justification_found": False,
            "evidence_count": len(evidence_texts)
        })

        continue

    print(
        f"Justification found: "
        f"{len(str(justification))} characters"
    )

    # ------------------------------------------------------------
    # RUN REVISED VERIFIER
    # ------------------------------------------------------------

    verification = verify_justification(
        justification=justification,
        evidence_texts=evidence_texts
    )

    revised_test_results.append({

        "claim_index": i,

        "claim_id": claim_id_value,

        "justification_found": True,

        "justification": justification,

        "evidence_count": len(evidence_texts),

        "n_sentences": (
            verification["n_sentences"]
        ),

        "n_unsupported": (
            verification["n_unsupported"]
        ),

        "unsupported_sentence_rate": (
            verification["unsupported_sentence_rate"]
        ),

        "mean_entailment": (
            verification["mean_entailment"]
        ),

        "claim_hallucinated": (
            verification["claim_hallucinated"]
        ),

        "sentence_results": (
            verification["sentences"]
        )
    })


# ================================================================
# SUMMARY
# ================================================================

valid_results = [
    r for r in revised_test_results
    if r.get("justification_found")
]

print("\n" + "=" * 80)
print("10-CLAIM REVISED VERIFICATION SUMMARY")
print("=" * 80)

print(
    "Claims successfully evaluated:",
    len(valid_results)
)

if valid_results:

    total_sentences = sum(
        r["n_sentences"]
        for r in valid_results
    )

    total_unsupported = sum(
        r["n_unsupported"]
        for r in valid_results
    )

    hallucinated_claims = sum(
        r["claim_hallucinated"]
        for r in valid_results
    )

    hallucination_rate = (
        hallucinated_claims /
        len(valid_results)
    )

    unsupported_sentence_rate = (
        total_unsupported /
        total_sentences
        if total_sentences > 0
        else 0
    )

    grounding = np.mean([
        r["mean_entailment"]
        for r in valid_results
    ])

    print(
        f"Claim hallucination rate: "
        f"{hallucination_rate:.4f}"
    )

    print(
        f"Unsupported sentence rate: "
        f"{unsupported_sentence_rate:.4f}"
    )

    print(
        f"Grounding score: "
        f"{grounding:.4f}"
    )

    print(
        f"Total sentences: "
        f"{total_sentences}"
    )

    print(
        f"Unsupported sentences: "
        f"{total_unsupported}"
    )

    print("\nPer-claim results:")

    for r in valid_results:

        print(
            f"Claim {r['claim_index']:02d} | "
            f"evidence={r['evidence_count']} | "
            f"sentences={r['n_sentences']} | "
            f"unsupported={r['n_unsupported']} | "
            f"entailment="
            f"{r['mean_entailment']:.3f} | "
            f"hallucinated="
            f"{r['claim_hallucinated']}"
        )

RUNNING REVISED VERIFICATION ON 10 CLAIMS

Processing claim 0 | claim_id = 95405
Evidence documents: 10
Justification found: 275 characters

Processing claim 1 | claim_id = 43167
Evidence documents: 10
Justification found: 157 characters

Processing claim 2 | claim_id = 66064
Evidence documents: 10
Justification found: 203 characters

Processing claim 3 | claim_id = 27959
Evidence documents: 10
Justification found: 332 characters

Processing claim 4 | claim_id = 74735
Evidence documents: 10
Justification found: 216 characters

Processing claim 5 | claim_id = 29100
Evidence documents: 10
Justification found: 228 characters

Processing claim 6 | claim_id = 51863
Evidence documents: 10
Justification found: 293 characters

Processing claim 7 | claim_id = 31488
Evidence documents: 10
Justification found: 251 characters

Processing claim 8 | claim_id = 65838
Evidence documents: 10
Justification found: 265 characters

Processing claim 9 | claim_id = 58416
Evidence documents: 10
Justification 

In [16]:
import inspect

print(inspect.signature(get_evidence_texts_proposed))
print()
print(inspect.getsource(get_evidence_texts_proposed))

(claim_id)

def get_evidence_texts_proposed(claim_id):
    ers_ranked = claim_id_to_ers.get(claim_id, [])
    return [
        evidence_corpus[item['corpus_idx']]['text']
        for item in ers_ranked[:MAX_EVIDENCE_PER_CLAIM]
        if item['corpus_idx'] < len(evidence_corpus)
    ]



In [17]:
print(claims[0])

{'claim_id': 95405, 'claim': 'William Dering is first recorded as a dancing master in Philadelphia, he is said to have been respected as well for his talents on the French horn and in 1737 he moved to Gloucester County which was founded in 1651.', 'label': 'NOT ENOUGH INFO', 'gold_evidence': [{'ev_id': 'William Dering_sentence_2', 'text': 'Dering is first recorded as a dancing master in [[Philadelphia,_Pennsylvania|Philadelphia, Pennsylvania]], from 1735 to 1736.', 'type': 'sentence', 'corpus_idx': 612}, {'ev_id': 'William Dering_sentence_3', 'text': 'A good conversationalist, he is said to have been respected as well for his talents on the French horn; his school also offered lessons in "Reading, Writing, Dancing, Plain Work, Marking, Embroidery, and several other Works: where Likewise young Ladies and Gentlemen may be instructed in the French."', 'type': 'sentence', 'corpus_idx': 2034}, {'ev_id': 'William Dering_sentence_5', 'text': 'Dering had moved to [[Gloucester_County,_Virginia|

In [19]:
# ================================================================
# DIAGNOSTIC — INSPECT SENTENCE-LEVEL NLI RESULTS
# ================================================================

for r in valid_results:

    print("\n" + "=" * 100)

    print(
        f"CLAIM {r['claim_index']} "
        f"(ID={r['claim_id']})"
    )

    print("-" * 100)

    print("JUSTIFICATION:")
    print(r["justification"])

    print("\nSENTENCE-LEVEL RESULTS:")

    for j, s in enumerate(r["sentence_results"]):

        print(f"\nSentence {j + 1}:")

        print(
            "Sentence:",
            s.get("sentence")
        )

        print(
            "Entailment:",
            s.get("entailment")
        )

        print(
            "Supported:",
            s.get("supported")
        )

        print(
            "Best evidence:",
            s.get("best_evidence")
        )


CLAIM 0 (ID=95405)
----------------------------------------------------------------------------------------------------
JUSTIFICATION:
The claim provides evidence that supports the statement about William Dering's career as a dancing master and his proficiency on the French horn. Additionally, it mentions the year he moved to Gloucester County, which was indeed founded in 1651, further supporting the claim.

SENTENCE-LEVEL RESULTS:

Sentence 1:
Sentence: The claim provides evidence that supports the statement about William Dering's career as a dancing master and his proficiency on the French horn.
Entailment: None
Supported: True
Best evidence: A good conversationalist, he is said to have been respected as well for his talents on the French horn; his school also offered lessons in "Reading, Writing, Dancing, Plain Work, Marking, Embroidery, and several other Works: where Likewise young Ladies and Gentlemen may be instructed in the French."

Sentence 2:
Sentence: Additionally, it menti

In [20]:
print(claims[3])

{'claim_id': 27959, 'claim': 'Adductor hiatus is associated with nine structures, seven of which enter and leave through hiatus.', 'label': 'REFUTES', 'gold_evidence': [{'ev_id': 'Adductor hiatus_sentence_8', 'text': 'Four structures are associated with the adductor hiatus.', 'type': 'sentence', 'corpus_idx': 2065}]}


In [21]:
# ================================================================
# DIAGNOSTIC — INSPECT ERS MAPPING FOR CLAIM 27959
# ================================================================

claim_id = 27959

print("=" * 100)
print("CLAIM")
print("=" * 100)

print(claims[3])

print("\n" + "=" * 100)
print("GOLD EVIDENCE")
print("=" * 100)

gold = claims[3]["gold_evidence"]

for item in gold:
    print("\nev_id:", item["ev_id"])
    print("corpus_idx:", item["corpus_idx"])
    print("text:", item["text"])


print("\n" + "=" * 100)
print("PROPOSED ERS ENTRIES")
print("=" * 100)

ers = claim_id_to_ers.get(claim_id, [])

print("Number of ERS entries:", len(ers))

for rank, item in enumerate(ers[:10], 1):

    print(f"\nRank {rank}")
    print("ERS item:", item)

    idx = item.get("corpus_idx")

    if idx is not None and idx < len(evidence_corpus):

        print("CORPUS TEXT:")
        print(evidence_corpus[idx]["text"])

    else:

        print("INVALID CORPUS INDEX")


print("\n" + "=" * 100)
print("EXPECTED GOLD CORPUS ENTRY")
print("=" * 100)

gold_idx = gold[0]["corpus_idx"]

print("Gold corpus_idx:", gold_idx)
print("Corpus length:", len(evidence_corpus))
print("Corpus text:")
print(evidence_corpus[gold_idx]["text"])

CLAIM
{'claim_id': 27959, 'claim': 'Adductor hiatus is associated with nine structures, seven of which enter and leave through hiatus.', 'label': 'REFUTES', 'gold_evidence': [{'ev_id': 'Adductor hiatus_sentence_8', 'text': 'Four structures are associated with the adductor hiatus.', 'type': 'sentence', 'corpus_idx': 2065}]}

GOLD EVIDENCE

ev_id: Adductor hiatus_sentence_8
corpus_idx: 2065
text: Four structures are associated with the adductor hiatus.

PROPOSED ERS ENTRIES
Number of ERS entries: 20

Rank 1
ERS item: {'corpus_idx': 2065, 'ers': 0.97213, 'relevance': 1.0, 'consistency': 0.81418}
CORPUS TEXT:
Four structures are associated with the adductor hiatus.

Rank 2
ERS item: {'corpus_idx': 1425, 'ers': 0.22302, 'relevance': 0.14239, 'consistency': 0.67995}
CORPUS TEXT:
Artists featured on the compilation album include [[Ulver|Ulver]], [[Nest_(band)|Nest]], [[Syven|Syven]], [[Les_Discrets|Les Discrets]], [[Neun_Welten|Neun Welten]], [[Nucleus_Torn|Nucleus Torn]], [[Orplid|Orplid]], 

In [22]:
# ================================================================
# DIAGNOSTIC — FULL NLI PROBABILITIES
# ================================================================

claim_id = 27959

claim_text = claims[3]["claim"]

gold_evidence = claims[3]["gold_evidence"][0]["text"]

print("=" * 80)
print("CLAIM")
print("=" * 80)
print(claim_text)

print("\n" + "=" * 80)
print("GOLD EVIDENCE")
print("=" * 80)
print(gold_evidence)

inputs = nli_tokenizer(
    gold_evidence,
    claim_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(nli_model.device)

with torch.no_grad():
    logits = nli_model(**inputs).logits

probs = logits.softmax(dim=-1).squeeze().cpu().numpy()

print("\n" + "=" * 80)
print("NLI PROBABILITIES")
print("=" * 80)

for idx, prob in enumerate(probs):
    print(
        f"{idx}: "
        f"{nli_model.config.id2label[idx]} = "
        f"{prob:.4f}"
    )

CLAIM
Adductor hiatus is associated with nine structures, seven of which enter and leave through hiatus.

GOLD EVIDENCE
Four structures are associated with the adductor hiatus.

NLI PROBABILITIES
0: contradiction = 0.9990
1: neutral = 0.0010
2: entailment = 0.0001


In [23]:
# ================================================================
# DIAGNOSTIC — FULL NLI BREAKDOWN FOR THE 10-CLAIM EXPERIMENT
# ================================================================

def get_nli_probabilities(premise, hypothesis):

    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(nli_model.device)

    with torch.no_grad():
        logits = nli_model(**inputs).logits

    probs = logits.softmax(dim=-1).squeeze().cpu().numpy()

    return {
        "contradiction": float(probs[0]),
        "neutral": float(probs[1]),
        "entailment": float(probs[2])
    }


print("=" * 100)
print("FULL NLI DIAGNOSTIC — 10 CLAIMS")
print("=" * 100)

for r in valid_results:

    print("\n" + "=" * 100)
    print(
        f"CLAIM {r['claim_index']} "
        f"(ID={r['claim_id']})"
    )
    print("=" * 100)

    justification = r["justification"]

    sentences = sent_tokenize(
        strip_citation_markers(justification)
    )

    sentences = [
        s.strip()
        for s in sentences
        if len(s.strip()) > 15
    ]

    evidence_texts = get_evidence_texts_proposed(
        r["claim_id"]
    )

    for i, sentence in enumerate(sentences):

        print(f"\n--- Sentence {i + 1} ---")

        print("JUSTIFICATION:")
        print(sentence)

        best = None

        for ev_idx, evidence in enumerate(evidence_texts):

            scores = get_nli_probabilities(
                premise=evidence,
                hypothesis=sentence
            )

            if (
                best is None
                or scores["entailment"] > best["scores"]["entailment"]
            ):
                best = {
                    "ev_idx": ev_idx,
                    "evidence": evidence,
                    "scores": scores
                }

        print("\nBEST EVIDENCE:")
        print(best["evidence"][:500])

        print("\nNLI PROBABILITIES:")
        print(
            f"Contradiction: "
            f"{best['scores']['contradiction']:.4f}"
        )
        print(
            f"Neutral:       "
            f"{best['scores']['neutral']:.4f}"
        )
        print(
            f"Entailment:    "
            f"{best['scores']['entailment']:.4f}"
        )

FULL NLI DIAGNOSTIC — 10 CLAIMS

CLAIM 0 (ID=95405)

--- Sentence 1 ---
JUSTIFICATION:
The claim provides evidence that supports the statement about William Dering's career as a dancing master and his proficiency on the French horn.

BEST EVIDENCE:
A good conversationalist, he is said to have been respected as well for his talents on the French horn; his school also offered lessons in "Reading, Writing, Dancing, Plain Work, Marking, Embroidery, and several other Works: where Likewise young Ladies and Gentlemen may be instructed in the French."

NLI PROBABILITIES:
Contradiction: 0.0001
Neutral:       0.0479
Entailment:    0.9519

--- Sentence 2 ---
JUSTIFICATION:
Additionally, it mentions the year he moved to Gloucester County, which was indeed founded in 1651, further supporting the claim.

BEST EVIDENCE:
Dering had moved to [[Gloucester_County,_Virginia|Gloucester County, Virginia]] by 1737.

NLI PROBABILITIES:
Contradiction: 0.0009
Neutral:       0.9985
Entailment:    0.0006

CLAIM 1